In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\Lenovo\Desktop\Parkinson's Disease\parkinsons.data")
print(df.shape)
#print(df.head())
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate Rows:", df.duplicated().sum())
print("\nTarget Balance:", df['status'].value_counts(normalize = True))

(195, 24)

Missing values:
 Series([], dtype: int64)

Duplicate Rows: 0

Target Balance: status
1    0.753846
0    0.246154
Name: proportion, dtype: float64


In [7]:
#Extracting patient groups

df['subject_id'] = df['name'].apply(lambda x: x.split('_')[2])
print("Unique subjects:", df['subject_id'].nunique(), "| Total rows:", len(df))

X = df.drop(columns=['name', 'status', 'subject_id'])
y = df['status']
groups = df['subject_id']

Unique subjects: 32 | Total rows: 195


In [8]:
from scipy import stats

z_scores = np.abs(stats.zscore(X))
outlier_rows = (z_scores > 4).any(axis=1)
print(f"Rows with |z|>4 in any feature: {outlier_rows.sum()}")

Rows with |z|>4 in any feature: 8


In [10]:
corr = X.corr()

In [11]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))

Train class balance:
 status
1    0.763158
0    0.236842
Name: proportion, dtype: float64
Test class balance:
 status
1    0.72093
0    0.27907
Name: proportion, dtype: float64


In [12]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report

gkf = GroupKFold(n_splits=5)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight='balanced', random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, random_state=42
    ),
}

results = {}

for name, model in models.items():
    all_y_true, all_y_pred = [], []
    
    for train_i, test_i in gkf.split(X, y, groups=groups):
        X_tr, X_te = X.iloc[train_i], X.iloc[test_i]
        y_tr, y_te = y.iloc[train_i], y.iloc[test_i]
        
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        
        all_y_true.extend(y_te)
        all_y_pred.extend(y_pred)
    
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print(classification_report(all_y_true, all_y_pred, target_names=['Healthy (0)', 'Diseased (1)']))
    results[name] = (all_y_true, all_y_pred)


Random Forest
              precision    recall  f1-score   support

 Healthy (0)       0.48      0.33      0.40        48
Diseased (1)       0.80      0.88      0.84       147

    accuracy                           0.75       195
   macro avg       0.64      0.61      0.62       195
weighted avg       0.72      0.75      0.73       195


Gradient Boosting
              precision    recall  f1-score   support

 Healthy (0)       0.53      0.40      0.45        48
Diseased (1)       0.82      0.88      0.85       147

    accuracy                           0.76       195
   macro avg       0.67      0.64      0.65       195
weighted avg       0.75      0.76      0.75       195

